# Official CODI endpoint TSV-C-inspired experiment

This notebook fits fixed rank-77 per-layer bases from the paper-accuracy official CODI checkpoint and runs the preregistered all-layer primary and layer-11 secondary held-out utility gates. It is designed for Kaggle **Save Version → Save & Run All** with Internet and a T4-class GPU enabled.

The original TSV-C method operates on weight-difference matrices. This experiment adapts its truncated-SVD principle to teacher-minus-student endpoint activation residuals. A positive gate authorizes a later training study; this notebook never trains or overwrites the official checkpoint.

## 1. Configuration

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Replace with the immutable commit printed by setup.
REPO_DIR = "/kaggle/working/latent-reasoning"

REPRODUCTION_SUMMARY_INPUT = ""  # Optional exact passed summary.json.
RESUME_INPUT = ""  # Optional attached prior endpoint-TSVC export root.
RUN_REPRODUCTION_GATE_IF_MISSING = True
RUN_SMOKE = True
RUN_FULL_CALIBRATION = True
RUN_ALL_LAYER_UTILITY = True
RUN_LAYER11_UTILITY = True

CALIBRATION_EXAMPLES = 5000
UPDATE_EXAMPLES = 256
VALIDATION_EXAMPLES = 256
CALIBRATION_BATCH_SIZE = 16
UTILITY_BATCH_SIZE = 4
SAMPLING_SEED = 11
RANDOM_BASIS_SEED = 20260803
RANK = 77
RELATIVE_UPDATE_NORM = 1e-4
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-endpoint-tsvc"

## 2. Install and pin the repository

In [ ]:
import datetime, hashlib, json, os, pathlib, shutil, subprocess, sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL SAVE VERSION RUN:", commit)

## 3. Hardware and implementation checks

In [ ]:
import torch, transformers
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__, "Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "capability:", capability)
assert capability >= (7, 0), "This Kaggle PyTorch build requires a T4-class or newer GPU"
subprocess.run([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_official_codi.py",
    "tests/test_official_codi_target_utility.py",
    "tests/test_endpoint_tsvc.py",
    "tests/test_official_codi_endpoint_tsvc_analysis.py",
], cwd=REPO_DIR, check=True)

## 4. Durable paths, logging, and optional resume

In [ ]:
OUTPUT_ROOT = repo / "outputs" / "official_codi_endpoint_tsvc"
REPORT_ROOT = repo / "reports" / "official_codi_endpoint_tsvc"
LOG_ROOT = repo / "logs" / "official_codi_endpoint_tsvc"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (OUTPUT_ROOT, REPORT_ROOT, LOG_ROOT, VALIDATION_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(command, log_name):
    log_path = LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(map(str, command))} ===\n")
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
        log.flush()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}; inspect {log_path}")
    return log_path

if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(resume_root.rglob("official_codi_endpoint_tsvc/calibration_seed11/run_manifest.json"))
    assert manifests, "No endpoint TSV-C calibration manifest found in RESUME_INPUT"
    request_hashes = {json.loads(path.read_text()).get("request_sha256") for path in manifests}
    assert len(request_hashes) == 1, f"Incompatible duplicate resume trees: {manifests}"
    source_root = sorted({path.parents[1] for path in manifests}, key=lambda p: (len(p.parts), p.as_posix()))[0]
    shutil.copytree(source_root, OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored endpoint TSV-C outputs from:", source_root)
else:
    print("Starting without prior endpoint TSV-C outputs")

## 5. Locate or create the official CODI reproduction gate

In [ ]:
EXPECTED_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"
def passed_summary(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    gate = payload.get("accuracy_gate", payload.get("gate"))
    status = gate.get("status") if isinstance(gate, dict) else gate
    count = payload.get("evaluated_counts", {}).get("gsm8k")
    revision = payload.get("checkpoint_revision")
    return status == "passed" and count == 1319 and (revision in {None, EXPECTED_REVISION})

if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT)
    assert passed_summary(REPRODUCTION_SUMMARY)
else:
    candidates = [p for p in pathlib.Path("/kaggle/input").rglob("summary.json") if passed_summary(p)]
    candidates += [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    if not candidates:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted([sys.executable, "-u", "-m", "src.eval.official_codi", "--config", "configs/official_codi_gpt2.yaml", "--datasets", "gsm8k", "--limit", "0", "--device", "cuda", "--output-dir", str(VALIDATION_ROOT)], "official_codi_gsm8k_gate.log")
        candidates = [p for p in VALIDATION_ROOT.rglob("summary.json") if passed_summary(p)]
    assert candidates, "A passed full-GSM8K official CODI summary is required"
    REPRODUCTION_SUMMARY = sorted(candidates, key=lambda p: p.as_posix())[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 6. Smoke test the complete path

In [ ]:
def collection_command(root, calibration, update, validation, batch):
    return [sys.executable, "-u", "scripts/collect_official_codi_endpoint_tsvc.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--output-dir", str(root), "--calibration-examples", str(calibration), "--update-examples", str(update), "--validation-examples", str(validation), "--batch-size", str(batch), "--save-every", "8", "--rank", str(RANK), "--sampling-seed", str(SAMPLING_SEED), "--random-basis-seed", str(RANDOM_BASIS_SEED), "--precision", PRECISION, "--device", "cuda"]

def utility_command(root, basis, scope, bootstrap_samples=BOOTSTRAP_SAMPLES):
    return [sys.executable, "-u", "scripts/run_official_codi_endpoint_tsvc_utility.py", "--config", "configs/official_codi_gpt2.yaml", "--reproduction-summary", str(REPRODUCTION_SUMMARY), "--basis", str(basis), "--output-dir", str(root), "--scope", scope, "--batch-size", str(UTILITY_BATCH_SIZE), "--relative-update-norm", str(RELATIVE_UPDATE_NORM), "--precision", PRECISION, "--device", "cuda", "--seed", str(SAMPLING_SEED), "--bootstrap-samples", str(bootstrap_samples), "--bootstrap-seed", str(BOOTSTRAP_SEED)]

SMOKE_CALIBRATION = OUTPUT_ROOT / "smoke_seed11" / "calibration"
SMOKE_UTILITY = OUTPUT_ROOT / "smoke_seed11" / "endpoint_all_layers"
if RUN_SMOKE:
    run_persisted(collection_command(SMOKE_CALIBRATION, 16, 8, 8, 8), "smoke_calibration.log")
    run_persisted(utility_command(SMOKE_UTILITY, SMOKE_CALIBRATION / "basis.pt", "endpoint_all_layers", 500), "smoke_utility.log")
    assert json.loads((SMOKE_UTILITY / "run_manifest.json").read_text())["state"] == "complete"
    print("Smoke path complete. Its gate is diagnostic only.")
else:
    print("Smoke test skipped")

## 7. Fit or resume the preregistered 5,000-example bases

In [ ]:
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration_seed11"
BASIS_PATH = CALIBRATION_ROOT / "basis.pt"
if RUN_FULL_CALIBRATION:
    command = collection_command(CALIBRATION_ROOT, CALIBRATION_EXAMPLES, UPDATE_EXAMPLES, VALIDATION_EXAMPLES, CALIBRATION_BATCH_SIZE)
    command[command.index("--save-every") + 1] = "500"
    run_persisted(command, "calibration_n5000_seed11.log")
assert BASIS_PATH.is_file(), f"Missing preregistered basis: {BASIS_PATH}"
calibration_manifest = json.loads((CALIBRATION_ROOT / "run_manifest.json").read_text())
assert calibration_manifest["state"] == "complete"
assert calibration_manifest["calibration_examples"] == CALIBRATION_EXAMPLES
assert calibration_manifest["rank"] == 77
print("Basis SHA256:", calibration_manifest["basis_sha256"])

## 8. Run the primary all-layer and secondary layer-11 utility scopes

In [ ]:
ALL_LAYER_ROOT = OUTPUT_ROOT / "utility_seed11" / "endpoint_all_layers"
LAYER11_ROOT = OUTPUT_ROOT / "utility_seed11" / "endpoint_layer11"
if RUN_ALL_LAYER_UTILITY:
    run_persisted(utility_command(ALL_LAYER_ROOT, BASIS_PATH, "endpoint_all_layers"), "utility_all_layers_seed11.log")
if RUN_LAYER11_UTILITY:
    run_persisted(utility_command(LAYER11_ROOT, BASIS_PATH, "endpoint_layer11"), "utility_layer11_seed11.log")
for root in (ALL_LAYER_ROOT, LAYER11_ROOT):
    manifest = json.loads((root / "run_manifest.json").read_text())
    assert manifest["state"] == "complete"
    assert len(manifest["completed_batches"]) == 64
print("Both preregistered utility scopes are complete")

## 9. Combine the confirmatory decision

In [ ]:
from IPython.display import Markdown, display
REPORT_PATH = REPORT_ROOT / "official_codi_endpoint_tsvc_seed11.json"
run_persisted([sys.executable, "scripts/analyze_official_codi_endpoint_tsvc.py", "--all-layers", str(ALL_LAYER_ROOT), "--layer11", str(LAYER11_ROOT), "--output", str(REPORT_PATH)], "analyze_endpoint_tsvc_seed11.log")
report = json.loads(REPORT_PATH.read_text())
display(Markdown(REPORT_PATH.with_suffix(".md").read_text()))
print("FINAL STATUS:", report["status"])
print("TRAINING AUTHORIZED:", report["training_authorized"])

## 10. Build a checksummed durable export

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_endpoint_tsvc_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(OUTPUT_ROOT, export_repo / "outputs" / "official_codi_endpoint_tsvc")
shutil.copytree(REPORT_ROOT, export_repo / "reports" / "official_codi_endpoint_tsvc")
shutil.copytree(LOG_ROOT, export_repo / "logs" / "official_codi_endpoint_tsvc")
validation_export = export_repo / "outputs" / "official_codi_gpt2_reproduction"
validation_export.mkdir(parents=True, exist_ok=True)
shutil.copy2(REPRODUCTION_SUMMARY, validation_export / "summary.json")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "RUN_INSTRUCTIONS.txt").write_text("Attach this dataset and set RESUME_INPUT to its root to resume incomplete endpoint TSV-C batches.\n")
files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
lines = [f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(EXPORT_ROOT).as_posix()}" for path in files]
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(lines) + "\n")
all_files = [path for path in EXPORT_ROOT.rglob("*") if path.is_file()]
print("Export root:", EXPORT_ROOT)
print("Files:", len(all_files), "Size MiB:", sum(path.stat().st_size for path in all_files) / 2**20)
print("Use Save Version with outputs enabled.")

## 11. Optional direct Kaggle Dataset upload

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(KAGGLE_DATASET_HANDLE, str(EXPORT_ROOT), version_notes=f"Official CODI endpoint TSV-C gate at {commit}")
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct upload skipped; Save Version with outputs enabled is sufficient.")

## Interpretation

Only `endpoint_all_layers` can authorize a later training study. A layer-11-only pass requires fresh confirmation. If both fail, close this fixed linear rank-77 endpoint definition rather than tuning the rank or gate on the observed results.